# FLUX outpainting — smoke (cheap load + one-sided 256px extend)

Fast path: static checks → CPU-only geometry/mask unit checks → publish PRIVATE → modular load →
a small `outpaint-flux-modular` run at low res / few steps. Full validation (all-sides extend,
bit-exact interior, seam metric) is in `e2e.ipynb`.

Runtime: A100 · `HUGGINGFACE_TOKEN` · accept **FLUX.1-dev** AND **FLUX.1-Fill-dev** licenses.


## 1 · Install + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf


In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))


## 2 · Static checks (the CI gate) + CPU-only geometry/mask checks

This runs the repo's own `scripts/check_pipeline.py` against the uploaded `block.py`, then exercises
the block's pure-CPU geometry (margin resolution, VAE rounding, keep-box erosion, mask coverage) with
**no GPU and no weights downloaded** — so a geometry regression is caught before any FLUX call.

In [ ]:
import glob, json, pathlib, subprocess, sys

# 2a. the repo's static gate, exactly as CI runs it
for f in glob.glob("*.py"):                     # block.py must be present next to this notebook
    print("file:", f)
r = subprocess.run([sys.executable, "check_pipeline.py", "."], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert "OK" in (r.stdout + r.stderr), "static checks failed"

# 2b. CPU-only geometry/mask checks against the real block class
sys.path.insert(0, ".")
import block as outpaint_block
OB = outpaint_block.OutpaintBlock
G, K = OB._canvas_geometry, OB._keep_box

# per-side margins are honoured exactly; VAE-rounding slack lands on the trailing side
assert G(100, 60, 50, 50, 20, 20, None, None) == (200, 104, 50, 20)
assert G(101, 101, 50, 50, 50, 50, None, None) == (208, 208, 50, 50)   # 201 -> 208
assert G(64, 64, 256, 0, 0, 0, None, None) == (320, 64, 256, 0)        # left margin only
assert G(64, 64, 0, 0, 128, 0, None, None) == (64, 192, 0, 128)        # top margin only
# a 0-1 fraction is a fraction of the source size on that axis
assert G(100, 100, 0.5, 0, 0, 0, None, None) == (152, 104, 50, 0)
# target size centers, and is mutually exclusive with margins
assert G(64, 64, None, None, None, None, 256, 256) == (256, 256, 96, 96)
for bad in [(64, 64, 8, 8, 8, 8, 256, 256), (64, 64, None, None, None, None, 32, 32)]:
    try:
        G(*bad); raise AssertionError(f"geometry accepted {bad}")
    except ValueError:
        pass
# keep-box erodes by the feather and clamps to a 2px core
assert K(10, 20, 100, 60, 8) == (18, 28, 84, 44)
assert K(10, 20, 100, 60, 0) == (10, 20, 100, 60)
assert K(0, 0, 8, 8, 1000) == (3, 3, 2, 2)
print("[OK] geometry + keep-box: margins honoured, VAE rounding, feather erosion")


## 3 · Publish PRIVATE (upload `block.py` + configs first)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); REPO = "remyxai/outpaint-flux-modular"
api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
for f in ["block.py", "modular_config.json", "modular_model_index.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
print("published:", api.list_repo_files(REPO))


## 4 · Load via `trust_remote_code` (assert the block class)

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained("remyxai/outpaint-flux-modular", trust_remote_code=True)
print("loaded block:", type(pipe.blocks).__name__)
assert type(pipe.blocks).__name__ == "OutpaintBlock", "unexpected block class"
pipe.load_components(dtype=DT); pipe.to(DEV)
print("components:", sorted(pipe.components.keys()))


## 5 · Smoke run — extend one side by 256px

Small source, few steps, **one** margin: the cheapest run that still exercises the canvas/mask/paste-back
path end to end.

In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display

src = Image.open("smoke_src.png").convert("RGB") if os.path.exists("smoke_src.png") else \
      Image.fromarray((np.random.default_rng(0).random((256, 256, 3)) * 255).astype(np.uint8))
src.save("smoke_src.png")

g = torch.Generator(DEV).manual_seed(0)
out = pipe(image=src, prompt="open sky and distant hills", left=256,
           num_inference_steps=8, guidance_scale=30, generator=g).images[0]
out.save("smoke_out.png")
print("source:", src.size, "-> output:", out.size)
assert out.size[0] == src.size[0] + 256 and out.size[1] == src.size[1], "output must be wider by the margin"
# the source sits at x=256; its interior is restored bit-exactly. mask_feather=8 also repaints the
# old right border, so bit-exactness is claimed for the eroded box, not the last 8px. The margin is
# horizontal only, so the paste origin is (x,y)=(256,0) and rows are NOT offset by the margin.
W, H, MARGIN, F = src.size[0], src.size[1], 256, 8
o, s = np.asarray(out), np.asarray(src)
assert np.array_equal(o[F:H - F, MARGIN + F:MARGIN + W - F], s[F:H - F, F:W - F]), \
    "interior must be untouched"
print(f"[MILESTONE] one-sided extend OK; interior bit-exact ({W - 2 * F}x{H - 2 * F} px)")
display(out)


## Verdict

`loaded block: OutpaintBlock` + an output exactly one margin wider + the original band bit-exact =
the smoke path passes. Run `e2e.ipynb` for the all-sides extend, the seam metric, and the no-op control.